## Question 6 — Multi-head attention

Each head reads one slice of the projected queries, keys, and values. The head outputs are concatenated and mixed by $W_O$.

**Imports.** NumPy builds the matrices and checks the result.

In [1]:
import numpy as np

**Attention from earlier questions.** The two-head check calls single-head self-attention, and each head uses stable softmax.

In [2]:
def stable_softmax(x, axis=-1):
    """Numerically stable softmax along one axis."""
    x = np.asarray(x, dtype=float)
    shifted = x - np.max(x, axis=axis, keepdims=True)
    exp_shifted = np.exp(shifted)
    return exp_shifted / np.sum(exp_shifted, axis=axis, keepdims=True)


def scaled_dot_product_attention(Q, K, V):
    """Q:(nq, dk), K:(nk, dk), V:(nk, dv) -> output (nq, dv), weights (nq, nk)."""
    Q, K, V = (np.asarray(z, dtype=float) for z in (Q, K, V))
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights


def self_attention(X, W_Q, W_K, W_V):
    """Project (n, d_model) X, then return attention output and weights."""
    Q, K, V = X @ W_Q, X @ W_K, X @ W_V
    return scaled_dot_product_attention(Q, K, V)


**One head.** Scale the scores, optionally hide future positions with $-\infty$, then apply softmax.

In [3]:
def attention_with_mask(Q, K, V, causal=False):
    """One head: Q (nq, dk), K (nk, dk), V (nk, dv)."""
    scores = (Q @ K.T) / np.sqrt(Q.shape[1])
    if causal:
        if Q.shape[0] != K.shape[0]:
            raise ValueError("Causal mask requires equal query/key lengths")
        future = np.triu(np.ones(scores.shape, dtype=bool), k=1)
        scores = np.where(future, -np.inf, scores)
    
    weights = stable_softmax(scores, axis=-1)
    return weights @ V, weights


**Multi-head attention.** Split the packed projections into `num_heads` slices, run each head, concatenate, then apply $W_O$.

In [4]:
def multi_head_attention_general(query_states, key_value_states, W_Q, W_K, W_V, W_O, num_heads, causal=False):
    """Packed-head attention: output (nq, d_model), weights (H, nq, nk)."""
    query_states = np.asarray(query_states, dtype=float)
    key_value_states = np.asarray(key_value_states, dtype=float)
    if query_states.ndim != 2 or key_value_states.ndim != 2 or query_states.shape[1] != key_value_states.shape[1]:
        raise ValueError("Input states must be 2D with the same model width")
    d_model = query_states.shape[1]
    if num_heads < 1 or d_model % num_heads:
        raise ValueError("num_heads must divide d_model")
    for W in (W_Q, W_K, W_V, W_O):
        if np.shape(W) != (d_model, d_model):
            raise ValueError("Packed projection matrices must be (d_model, d_model)")
    Q = query_states @ W_Q
    K = key_value_states @ W_K
    V = key_value_states @ W_V
    d_head = d_model // num_heads
    outputs, weights = [], []
    for h in range(num_heads):
        sl = slice(h * d_head, (h + 1) * d_head)
        head_output, head_weights = attention_with_mask(Q[:, sl], K[:, sl], V[:, sl], causal=causal)
        outputs.append(head_output)
        weights.append(head_weights)
    concatenated = np.concatenate(outputs, axis=-1)
    return concatenated @ W_O, np.stack(weights, axis=0)


**Self-attention.** Queries, keys, and values all come from $X$.

In [5]:
def multi_head_attention(X, W_Q, W_K, W_V, W_O, num_heads):
    """Self-attention: output (n, d_model), weights (H, n, n)."""
    return multi_head_attention_general(X, X, W_Q, W_K, W_V, W_O, num_heads)


**Check two separate heads.** On $I_2$, identity projections and swapped projections give the same weights. Their outputs are concatenated.

In [6]:
X6 = np.eye(2)
swap = np.array([[0.0, 1.0], [1.0, 0.0]])
o61, a61 = self_attention(X6, np.eye(2), np.eye(2), np.eye(2))
o62, a62 = self_attention(X6, swap, swap, np.eye(2))
concat6 = np.concatenate([o61, o62], axis=1)
print("head weights:\n", np.round(a61, 6))
print(np.round(a62, 6))
print("concatenated:\n", np.round(concat6, 6))
np.testing.assert_allclose(a61, a62)
print("Two-head check passed.")


head weights:
 [[0.669762 0.330238]
 [0.330238 0.669762]]
[[0.669762 0.330238]
 [0.330238 0.669762]]
concatenated:
 [[0.669762 0.330238 0.669762 0.330238]
 [0.330238 0.669762 0.330238 0.669762]]
Two-head check passed.


**Check the packed form.** With $d_{model}=8$ and 2 heads, the output is $3\times 8$ and the weights are $2\times 3\times 3$.

In [7]:
rng = np.random.default_rng(0)
X6_8 = rng.normal(size=(3, 8))
o6_8, a6_8 = multi_head_attention(X6_8, *([np.eye(8)] * 4), num_heads=2)
assert o6_8.shape == (3, 8) and a6_8.shape == (2, 3, 3)
assert np.all(np.isfinite(o6_8)) and np.all(np.isfinite(a6_8))
print("output", o6_8.shape, "weights", a6_8.shape)
print("Packed multi-head check passed.")


output (3, 8) weights (2, 3, 3)
Packed multi-head check passed.


(d) In training, different head parameters can learn different token relationships. This particular example does not demonstrate different relationships because both heads' dot products coincide. At fixed $d_{model}$, increasing head count reduces each head's width; it can create narrower heads, extra overhead, and redundant patterns, so more heads need not improve useful capacity.